# 01 — Exploration du dataset PaySim

Ce notebook explore les données brutes avant nettoyage.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os
sys.path.insert(0, '../src')
from load_data import load_paysim, get_basic_stats

%matplotlib inline
sns.set_theme(style='whitegrid')

In [ ]:
df = load_paysim()
print(df.shape)
df.head()

In [ ]:
stats = get_basic_stats(df)
for k, v in stats.items():
    print(f'{k}: {v}')

In [ ]:
# Distribution des types de transactions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['type'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Distribution des types de transactions')
axes[0].set_xlabel('Type')
axes[0].set_ylabel('Nombre')

df['isFraud'].value_counts().plot(kind='bar', ax=axes[1], color=['green', 'red'])
axes[1].set_title('Distribution fraude vs non-fraude')
axes[1].set_xlabel('isFraud')
axes[1].set_ylabel('Nombre')

plt.tight_layout()
plt.show()

In [ ]:
# Distribution des montants (log scale)
plt.figure(figsize=(10, 5))
sns.histplot(np.log1p(df['amount']), bins=50, kde=True)
plt.title('Distribution log(amount+1)')
plt.xlabel('log(amount + 1)')
plt.show()

In [ ]:
# Fraudes par type de transaction
fraud_by_type = df.groupby('type')['isFraud'].agg(['sum', 'mean'])
fraud_by_type.columns = ['fraud_count', 'fraud_rate']
fraud_by_type['fraud_rate'] = fraud_by_type['fraud_rate'].apply(lambda x: f'{x:.4%}')
print(fraud_by_type)

In [ ]:
# Valeurs manquantes
missing = df.isnull().sum()
print('Valeurs manquantes :')
print(missing[missing > 0] if missing.any() else 'Aucune')

In [ ]:
# Évolution temporelle
plt.figure(figsize=(14, 5))
tx_per_step = df.groupby('step').size()
fraud_per_step = df[df['isFraud']==1].groupby('step').size()

tx_per_step.plot(label='Total transactions', alpha=0.7)
fraud_per_step.plot(label='Fraudes', color='red', alpha=0.9)
plt.title('Transactions et fraudes par step temporel')
plt.legend()
plt.show()